# 1.0 — Data Acquisition

Download S&P 500 financial statements, daily prices, and macro indicators
from yfinance. Saves interim data to `data/interim/` for downstream use.

> **Note:** This notebook is exploratory — it fetches raw data and performs
> basic sanity checks. For reproducible pipeline execution, use `make all`.

In [ ]:
%load_ext autoreload
%autoreload 2
%cd ..

In [ ]:
import pandas as pd

from stock_data.config import EARNINGS_LAG_DAYS
from stock_data.dataset import (
    load_sp500_universe, get_active_symbols,
    fetch_quarterly_income, fetch_quarterly_balance_sheets,
    fetch_quarterly_cashflows, fetch_annual_income,
    reshape_statements, drop_sparse_pairs, pivot_statements,
    reshape_annual_income, download_prices, download_macro,
    compute_forward_returns, compute_realized_vol,
)

## Universe

In [ ]:
smp500_stocks = load_sp500_universe("data/raw/sp500_monthly.csv")
smp500_stocks.info()
smp500_stocks.head()

In [ ]:
current_sp500 = smp500_stocks[smp500_stocks["date_removed"].isna()]
symbols = get_active_symbols(current_sp500)
print(f"Number of currently active S&P 500 symbols: {len(symbols)}")
print(f"First 10 symbols: {symbols[:10]}")

## Quarterly Financial Statements

In [ ]:
quarterly_income_stmts, failed_symbols = fetch_quarterly_income(symbols)
combined = drop_sparse_pairs(reshape_statements(quarterly_income_stmts))
print(f"\nCombined table shape: {combined.shape}")
combined.head(20)

In [ ]:
features_raw = pivot_statements(combined)
print(f"Features raw: {features_raw.shape}")
features_raw.head()

In [ ]:
quarterly_balance_sheets, _ = fetch_quarterly_balance_sheets(symbols)
bs_raw = pivot_statements(reshape_statements(quarterly_balance_sheets))
print(f"Balance sheet: {bs_raw.shape}")

In [ ]:
quarterly_cashflows, _ = fetch_quarterly_cashflows(symbols)
cf_raw = pivot_statements(reshape_statements(quarterly_cashflows))
print(f"Cash flow: {cf_raw.shape}")

In [ ]:
annual_income_stmts, _ = fetch_annual_income(symbols)
annual_raw = reshape_annual_income(annual_income_stmts)
print(f"Annual raw: {annual_raw.shape}")

## Prices & Macro

In [ ]:
inc_symbols = combined.index.get_level_values("symbol").unique().tolist()
all_inc_dates = combined.index.get_level_values("date")
min_date = all_inc_dates.min() - pd.Timedelta(days=400)
max_date = all_inc_dates.max() + pd.Timedelta(days=120)

close_prices = download_prices(inc_symbols, min_date, max_date)

In [ ]:
macro_df = download_macro(min_date, max_date)
print(f"Macro features: {macro_df.shape}")
print(f"Columns: {macro_df.columns.tolist()}")
macro_df.tail()

## Forward Returns & Realized Volatility

In [ ]:
sym_date_pairs = combined.index.droplevel("item").unique()
returns_df = compute_forward_returns(close_prices, sym_date_pairs)
returns_df["return_quantile"] = returns_df.groupby("date")["next_q_return"].rank(pct=True)

print(f"Computed {len(returns_df)} returns with {EARNINGS_LAG_DAYS}-day earnings lag")
print(f"Unique symbols: {returns_df['symbol'].nunique()}")
returns_df.head(10)

In [ ]:
vol_df = compute_realized_vol(returns_df, close_prices)
returns_full = returns_df.merge(vol_df, on=["symbol", "date"], how="inner")
print(f"Returns + vol dataset: {len(returns_full)} rows")
returns_full.head()

## Save Interim Data

In [ ]:
from pathlib import Path

INTERIM = Path("data/interim")
INTERIM.mkdir(parents=True, exist_ok=True)

features_raw.to_parquet(INTERIM / "features_raw.parquet")
bs_raw.to_parquet(INTERIM / "bs_raw.parquet")
cf_raw.to_parquet(INTERIM / "cf_raw.parquet")
annual_raw.to_parquet(INTERIM / "annual_raw.parquet")
close_prices.to_parquet(INTERIM / "close_prices.parquet")
macro_df.to_parquet(INTERIM / "macro_df.parquet")
returns_df.to_parquet(INTERIM / "returns_df.parquet")
returns_full.to_parquet(INTERIM / "returns_full.parquet")

print(f"Interim data saved to {INTERIM}/")
print(f"  features_raw: {features_raw.shape}")
print(f"  bs_raw:       {bs_raw.shape}")
print(f"  cf_raw:       {cf_raw.shape}")
print(f"  annual_raw:   {annual_raw.shape}")
print(f"  close_prices: {close_prices.shape}")
print(f"  macro_df:     {macro_df.shape}")
print(f"  returns_full: {returns_full.shape}")